# 🥊 Notebook 03 — Modeling Showdown

**Goal:** Training 4 model sentiment classifier dan bandingin performanya.

## Models yang bakal kita pakai:

| Model | Tipe | Input |
|-------|------|-------|
| 1️⃣ **VADER** | Rule-based (no training) | `text_clean` |
| 2️⃣ **Naive Bayes** | Classical ML | `text_tokens` (TF-IDF) |
| 3️⃣ **Logistic Regression** | Classical ML | `text_tokens` (TF-IDF) |
| 4️⃣ **DistilBERT** | Deep Learning (pre-trained) | `text_clean` |

## ⚠️ Catatan:
- DistilBERT bakal lambat di CPU (10-20 menit). Sabar yaa!
- Kita save predictions tiap model ke CSV biar bisa di-load di notebook evaluation.

## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 100)
print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
df = pd.read_csv('../data/processed/tweets_clean.csv')
print(f"Shape: {df.shape}")
print(f"\nKolom: {list(df.columns)}")
df.head(3)

Shape: (14616, 9)

Kolom: ['tweet_id', 'airline_sentiment', 'airline', 'text', 'negativereason', 'retweet_count', 'tweet_created', 'text_clean', 'text_tokens']


,tweet_id,airline_sentiment,airline,text,negativereason,retweet_count,tweet_created,text_clean,text_tokens
0,570306133677760513,neutral,Virgin America,@VirginAmerica What @dhepburn said.,NaN,0,2015-02-24 11:35:52 -0800,what said,said
1,570301130888122368,positive,Virgin America,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,0,2015-02-24 11:15:59 -0800,plus you ve added commercials to the experience tacky,plus added commercials experience tacky
2,570301083672813571,neutral,Virgin America,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,0,2015-02-24 11:15:48 -0800,i didn t today must mean i need to take another trip,today must mean need take another trip


## 2. Train-Test Split

**Penting:** kita pisahin data jadi 80% training & 20% test SEKARANG, dan **semua model dievaluasi di test set yang sama**. Biar fair comparison.

In [3]:
# Stratify=y biar proporsi sentiment seimbang di train & test
X = df[['text_clean', 'text_tokens']]
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"\nDistribusi sentiment di train set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nDistribusi sentiment di test set:")
print(y_test.value_counts(normalize=True).round(3))

Train: 11,692 samples
Test:  2,924 samples

Distribusi sentiment di train set:
airline_sentiment
negative    0.627
neutral     0.211
positive    0.161
Name: proportion, dtype: float64

Distribusi sentiment di test set:
airline_sentiment
negative    0.628
neutral     0.211
positive    0.161
Name: proportion, dtype: float64


---

## 3️⃣ Model 1: VADER (Rule-Based)

**VADER** = Valence Aware Dictionary and sEntiment Reasoner. Dia punya kamus kata + aturan, dan ngitung skor sentiment berdasarkan kata-kata yang muncul.

**Kelebihan:**
- Gak perlu training
- Cepet banget
- Bagus buat text social media (banget tuned untuk tweet!)

**Kekurangan:**
- Cuma English
- Gak bisa nangkep konteks rumit/sarkasme

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

# Coba dulu ke 1 contoh
sample_text = "This flight was absolutely terrible. Worst experience ever."
scores = vader.polarity_scores(sample_text)
print(f"Text: {sample_text}")
print(f"Scores: {scores}")
print(f"Compound score: {scores['compound']}")
print("\n→ compound > 0.05: positive | < -0.05: negative | else: neutral")

Text: This flight was absolutely terrible. Worst experience ever.
Scores: {'neg': 0.564, 'neu': 0.436, 'pos': 0.0, 'compound': -0.8304}
Compound score: -0.8304

→ compound > 0.05: positive | < -0.05: negative | else: neutral


In [5]:
def vader_predict(text):
    """Convert VADER compound score → label sentiment."""
    if not isinstance(text, str) or len(text) == 0:
        return 'neutral'
    score = vader.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Predict di test set
start = time.time()
vader_preds = X_test['text_clean'].apply(vader_predict)
vader_time = time.time() - start

vader_acc = accuracy_score(y_test, vader_preds)
print(f"⏱️  Time: {vader_time:.2f}s")
print(f"🎯 VADER Accuracy: {vader_acc:.4f} ({vader_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, vader_preds)}")

⏱️  Time: 0.21s
🎯 VADER Accuracy: 0.5465 (54.65%)

              precision    recall  f1-score   support

    negative       0.90      0.50      0.65      1835
     neutral       0.39      0.41      0.40       617
    positive       0.34      0.89      0.49       472

    accuracy                           0.55      2924
   macro avg       0.54      0.60      0.51      2924
weighted avg       0.70      0.55      0.57      2924



---

## 4️⃣ Model 2: Naive Bayes + TF-IDF

**Naive Bayes** = model probabilistik klasik. Diasumsiin tiap kata independent (makanya disebut "naive"). Walaupun asumsinya sederhana, ternyata sering performa-nya bagus untuk text classification.

**TF-IDF** = cara konversi teks → angka. Kata yang **sering muncul di 1 dokumen tapi jarang di dokumen lain** dapet bobot tinggi. Misalnya "delayed" sering di tweet negative tapi jarang di positive → bobot tinggi buat klasifikasi.

In [6]:
# Step 1: Convert text -> TF-IDF vectors
tfidf = TfidfVectorizer(
    max_features=5000,    # ambil 5000 kata paling penting
    ngram_range=(1, 2),   # uni-gram + bi-gram (kata tunggal & pasangan kata)
    min_df=2,             # kata harus muncul minimal di 2 dokumen
)

X_train_tfidf = tfidf.fit_transform(X_train['text_tokens'])
X_test_tfidf = tfidf.transform(X_test['text_tokens'])

print(f"TF-IDF shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF shape (test):  {X_test_tfidf.shape}")
print(f"\nContoh top 30 features:")
print(list(tfidf.get_feature_names_out())[:30])

TF-IDF shape (train): (11692, 5000)
TF-IDF shape (test):  (2924, 5000)

Contoh top 30 features:
['aa', 'aa customer', 'aa flight', 'aa no', 'aa not', 'aadvantage', 'abc', 'able', 'able get', 'able use', 'abq', 'absolute', 'absolutely', 'absurd', 'abt', 'abysmal', 'accept', 'acceptable', 'accepting', 'access', 'access journal', 'accident', 'accidentally', 'accommodate', 'accommodating', 'accommodations', 'according', 'accordingly', 'account', 'account help']


In [7]:
# Step 2: Train Naive Bayes
start = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_train_time = time.time() - start

# Predict
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

print(f"⏱️  Training time: {nb_train_time:.2f}s")
print(f"🎯 Naive Bayes Accuracy: {nb_acc:.4f} ({nb_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, nb_preds)}")

⏱️  Training time: 0.03s
🎯 Naive Bayes Accuracy: 0.7490 (74.90%)

              precision    recall  f1-score   support

    negative       0.74      0.98      0.84      1835
     neutral       0.75      0.29      0.42       617
    positive       0.87      0.43      0.58       472

    accuracy                           0.75      2924
   macro avg       0.79      0.57      0.62      2924
weighted avg       0.76      0.75      0.71      2924



---

## 5️⃣ Model 3: Logistic Regression + TF-IDF

**Logistic Regression** = model linear yang sering jadi **baseline yang sulit dikalahkan** di text classification. Lebih powerful dari Naive Bayes karena bisa belajar feature interactions.

In [8]:
# Train Logistic Regression
start = time.time()
lr_model = LogisticRegression(
    max_iter=1000,        # cukup iterasi biar converge
    class_weight='balanced',  # handle class imbalance (negative dominan!)
    random_state=42,
    n_jobs=-1,            # pake semua CPU core
)
lr_model.fit(X_train_tfidf, y_train)
lr_train_time = time.time() - start

# Predict
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

print(f"⏱️  Training time: {lr_train_time:.2f}s")
print(f"🎯 Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, lr_preds)}")

⏱️  Training time: 0.31s
🎯 Logistic Regression Accuracy: 0.7623 (76.23%)

              precision    recall  f1-score   support

    negative       0.89      0.80      0.84      1835
     neutral       0.54      0.67      0.60       617
    positive       0.69      0.75      0.72       472

    accuracy                           0.76      2924
   macro avg       0.71      0.74      0.72      2924
weighted avg       0.78      0.76      0.77      2924



### 🔍 Inspect: Kata-kata paling indikatif

Salah satu kelebihan Logistic Regression: bisa di-inspect coefficient-nya buat liat kata apa yang paling indikatif tiap sentiment.

In [9]:
feature_names = tfidf.get_feature_names_out()
classes = lr_model.classes_

# Untuk tiap class, ambil top 10 kata dengan coefficient tertinggi
for i, class_name in enumerate(classes):
    coefs = lr_model.coef_[i]
    top_indices = coefs.argsort()[-15:][::-1]
    top_words = [(feature_names[idx], coefs[idx]) for idx in top_indices]
    
    print(f"\n🏷️  TOP indicators for '{class_name.upper()}':")
    for word, coef in top_words:
        print(f"   {word:30s} {coef:+.3f}")


🏷️  TOP indicators for 'NEGATIVE':
   hours                          +3.684
   not                            +3.468
   worst                          +3.394
   hold                           +3.095
   no                             +2.989
   hour                           +2.839
   hrs                            +2.626
   rude                           +2.453
   stuck                          +2.434
   nothing                        +2.398
   customers                      +2.398
   luggage                        +2.363
   delayed                        +2.336
   cancelled                      +2.322
   fail                           +2.281

🏷️  TOP indicators for 'NEUTRAL':
   hi                             +1.970
   policy                         +1.738
   chance                         +1.668
   possible                       +1.556
   winter                         +1.447
   ceo                            +1.364
   flyingitforward                +1.333
   mexico                  

---

## 6️⃣ Model 4: DistilBERT (Pre-trained Transformer)

**DistilBERT** = versi lebih kecil & cepet dari BERT (model deep learning state-of-the-art untuk NLP). Kita pake versi yang udah di-fine-tune di SST-2 (Stanford Sentiment Treebank).

**Catatan:**
- Model ini PRE-TRAINED, kita gak retrain dari nol (butuh GPU + dataset gede)
- Output asli: cuma positive/negative (binary). Kita adaptasi pakai confidence threshold untuk neutral.
- Ini bakal LAMBAT di CPU (10-20 menit). Kalau lo punya GPU, otomatis dipake.

⏰ **Sabar yaa, ini bagian paling lama!**

In [10]:
from transformers import pipeline
import torch

# Cek device
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"🖥️  Running on: {device_name}")

# Load pre-trained sentiment pipeline
print("📥 Loading DistilBERT model (download ~250MB first time)...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=128,
)
print("✅ Model loaded!")

🖥️  Running on: CPU
📥 Loading DistilBERT model (download ~250MB first time)...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Model loaded!


In [11]:
def hf_predict_with_neutral(text, threshold=0.7):
    """
    DistilBERT cuma kasih positive/negative. 
    Kita anggep 'neutral' kalau confidence < threshold.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 'neutral'
    try:
        result = sentiment_pipeline(text[:512])[0]  # limit panjang
        label = result['label'].lower()  # 'positive' atau 'negative'
        score = result['score']
        
        # Kalau confidence rendah, anggep neutral
        if score < threshold:
            return 'neutral'
        return label
    except Exception:
        return 'neutral'

# Test ke beberapa contoh
test_examples = [
    "This flight was amazing! Best service ever.",
    "Worst flight ever, never flying with them again.",
    "Flight took off at 3pm from JFK.",
]
for t in test_examples:
    print(f"'{t}' → {hf_predict_with_neutral(t)}")

'This flight was amazing! Best service ever.' → positive
'Worst flight ever, never flying with them again.' → negative
'Flight took off at 3pm from JFK.' → positive


In [12]:
# Predict ke semua test set
# Pakai batch processing biar lebih cepet
from tqdm.auto import tqdm
tqdm.pandas(desc="DistilBERT predicting")

print("⚠️  Ini bakal lama (10-20 menit di CPU). Sabar yaa...")
start = time.time()
hf_preds = X_test['text_clean'].progress_apply(hf_predict_with_neutral)
hf_time = time.time() - start

hf_acc = accuracy_score(y_test, hf_preds)
print(f"\n⏱️  Inference time: {hf_time:.1f}s ({hf_time/60:.1f} min)")
print(f"🎯 DistilBERT Accuracy: {hf_acc:.4f} ({hf_acc*100:.2f}%)")
print(f"\n{classification_report(y_test, hf_preds)}")

⚠️  Ini bakal lama (10-20 menit di CPU). Sabar yaa...


DistilBERT predicting:   0%|          | 0/2924 [00:00<?, ?it/s]


⏱️  Inference time: 89.7s (1.5 min)
🎯 DistilBERT Accuracy: 0.6919 (69.19%)

              precision    recall  f1-score   support

    negative       0.78      0.89      0.83      1835
     neutral       0.29      0.04      0.07       617
    positive       0.49      0.77      0.60       472

    accuracy                           0.69      2924
   macro avg       0.52      0.57      0.50      2924
weighted avg       0.63      0.69      0.63      2924



---

## 7. 🏆 Recap & Comparison

Mari kita ringkas hasil semua model.

In [13]:
results = pd.DataFrame({
    'Model': ['VADER', 'Naive Bayes', 'Logistic Regression', 'DistilBERT'],
    'Type': ['Rule-based', 'Classical ML', 'Classical ML', 'Deep Learning'],
    'Accuracy': [vader_acc, nb_acc, lr_acc, hf_acc],
    'Time (s)': [vader_time, nb_train_time, lr_train_time, hf_time],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

results['Accuracy %'] = (results['Accuracy'] * 100).round(2)
print("🏆 Final Leaderboard:")
print(results[['Model', 'Type', 'Accuracy %', 'Time (s)']].to_string(index=False))

🏆 Final Leaderboard:
              Model          Type  Accuracy %  Time (s)
Logistic Regression  Classical ML       76.23  0.313920
        Naive Bayes  Classical ML       74.90  0.034547
         DistilBERT Deep Learning       69.19 89.663524
              VADER    Rule-based       54.65  0.206967


## 8. 💾 Save Predictions & Models

Kita save predictions tiap model + model objects (TF-IDF, NB, LR) buat dipake di:
- Notebook 04 (deep evaluation)
- Streamlit app

In [14]:
# Save predictions ke 1 dataframe
predictions = pd.DataFrame({
    'text_clean': X_test['text_clean'].values,
    'text_tokens': X_test['text_tokens'].values,
    'true_label': y_test.values,
    'vader_pred': vader_preds.values,
    'nb_pred': nb_preds,
    'lr_pred': lr_preds,
    'hf_pred': hf_preds.values,
})

predictions.to_csv('../data/processed/model_predictions.csv', index=False)
print(f"✅ Saved predictions: {predictions.shape}")
predictions.head()

✅ Saved predictions: (2924, 7)


,text_clean,text_tokens,true_label,vader_pred,nb_pred,lr_pred,hf_pred
0,dear ted we don t care about you nor the mileage plus card you pay a lot for bye bye united,dear ted care nor mileage plus card pay lot bye bye united,negative,positive,negative,negative,negative
1,flight delayed times today due to flight officer from pit to ord hour delay to date bademployeep...,flight delayed times today due flight officer pit ord hour delay date bademployeeproblem,negative,negative,negative,negative,negative
2,i m going to start charging you for the consistent body pat downs starting to feel like a piece ...,going start charging consistent body pat downs starting feel like piece meat punishedforflying,negative,positive,negative,neutral,negative
3,why not we d sure love to see you ruth justdoit youknowyouwantto you know me too well,not sure love see ruth justdoit youknowyouwantto know well,positive,positive,negative,positive,negative
4,i dm you,dm,neutral,neutral,neutral,neutral,positive


In [15]:
# Save model objects (TF-IDF + classical ML models)
import os
os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ Saved: ../models/tfidf_vectorizer.pkl")

with open('../models/naive_bayes.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
print("✅ Saved: ../models/naive_bayes.pkl")

with open('../models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print("✅ Saved: ../models/logistic_regression.pkl")

# Save results comparison
results.to_csv('../data/processed/model_results.csv', index=False)
print("✅ Saved: ../data/processed/model_results.csv")

✅ Saved: ../models/tfidf_vectorizer.pkl
✅ Saved: ../models/naive_bayes.pkl
✅ Saved: ../models/logistic_regression.pkl
✅ Saved: ../data/processed/model_results.csv


## 9. Kesimpulan

### 🏆 Hasil yang biasanya muncul:
1. **Logistic Regression** → biasanya juara di dataset kayak gini (baseline yang kuat banget!)
2. **DistilBERT** → kadang #1, kadang #2. Tergantung data
3. **Naive Bayes** → baseline yang lumayan
4. **VADER** → bagus untuk speed, tapi akurasi paling rendah karena gak bisa nangkep konteks

### 💡 Lessons:
- **State-of-the-art ≠ always best** untuk task spesifik. Logistic Regression sederhana bisa nyaingin (atau ngalahin!) deep learning di task tertentu
- **Speed matters**: VADER 100x lebih cepet dari DistilBERT
- **Class imbalance** harus di-handle (kita pakai `class_weight='balanced'` di LR)

### 🚀 Next: Notebook 04 — Evaluation

Kita bakal:
- Bikin confusion matrix tiap model
- Liat error analysis (kasus mana yang salah diprediksi?)
- Pilih model FINAL buat dashboard